# Drug–Drug Interaction Analysis Pipeline

Author: **Mimi Adu-Serwaah, PharmD, MS**

A Python pipeline that screens for potential drug–drug interactions using the **DrugBank vocabulary**.
It standardizes drug-name data, matches free-text drug names to DrugBank entries (exact, synonym, then
partial match), detects pairwise interactions, and produces CSV reports and a network visualization for
clinical decision support.

**Stack:** Python (pandas, networkx, matplotlib, re)

### Data (not included)
This repo does **not** redistribute DrugBank data. Download the DrugBank vocabulary CSV from
[go.drugbank.com](https://go.drugbank.com) under its free academic license and place it at
`data/drugbank_vocabulary.csv`.

### Usage
1. Install dependencies: `pip install pandas networkx matplotlib`
2. Place the DrugBank vocabulary CSV at `data/drugbank_vocabulary.csv`
3. Run the notebook top to bottom. Outputs are written to `outputs/`.


In [ ]:
# DrugBank Drug Interaction Checker
# Author: Mimi Adu-Serwaah
# Purpose: Analyze potential drug-drug interactions using a DrugBank vocabulary CSV file.

import os
import re
from itertools import combinations

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx


# =========================
# 1. FILE PATHS
# =========================

DRUGBANK_CSV_PATH = os.path.join("data", "drugbank_vocabulary.csv")  # download from go.drugbank.com (free academic license); not redistributed here
OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# 2. LOAD DATA
# =========================

def load_drugbank_csv(file_path):
    """
    Load the DrugBank vocabulary CSV.
    """
    df = pd.read_csv(file_path)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df


df = load_drugbank_csv(DRUGBANK_CSV_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# =========================
# 3. STANDARDIZE DRUGBANK DATA
# =========================

def find_column(df, possible_names):
    """
    Find the first matching column from a list of possible names.
    """
    for name in possible_names:
        if name in df.columns:
            return name
    return None


name_col = find_column(df, ["name", "drug_name", "common_name"])
drugbank_id_col = find_column(df, ["drugbank_id", "drugbankid", "id"])
synonyms_col = find_column(df, ["synonyms", "synonym"])
description_col = find_column(df, ["description"])
cas_col = find_column(df, ["cas_number", "cas"])


if name_col is None:
    raise ValueError("Could not find a drug name column. Please inspect df.columns.")


def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9\s\-]", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value


df["clean_name"] = df[name_col].apply(clean_text)

if synonyms_col:
    df["clean_synonyms"] = df[synonyms_col].apply(clean_text)
else:
    df["clean_synonyms"] = ""

if description_col:
    df["clean_description"] = df[description_col].apply(clean_text)
else:
    df["clean_description"] = ""

print("Standardization complete.")
df[[name_col, "clean_name", "clean_synonyms"]].head()

In [ ]:
# =========================
# 4. DRUG NAME MATCHING
# =========================

def match_drug(drug_name, df):
    """
    Match user-entered drug name to DrugBank vocabulary.
    Searches exact name first, then synonyms, then partial matches.
    """
    query = clean_text(drug_name)

    exact = df[df["clean_name"] == query]
    if not exact.empty:
        return exact.iloc[0]

    synonym_match = df[df["clean_synonyms"].str.contains(query, na=False)]
    if not synonym_match.empty:
        return synonym_match.iloc[0]

    partial = df[df["clean_name"].str.contains(query, na=False)]
    if not partial.empty:
        return partial.iloc[0]

    return None


def resolve_drug_list(drug_list, df):
    """
    Resolve all input drugs to DrugBank entries.
    """
    results = []

    for drug in drug_list:
        match = match_drug(drug, df)

        if match is None:
            results.append({
                "input_drug": drug,
                "matched_name": None,
                "drugbank_id": None,
                "match_status": "No match found"
            })
        else:
            results.append({
                "input_drug": drug,
                "matched_name": match[name_col],
                "drugbank_id": match[drugbank_id_col] if drugbank_id_col else None,
                "match_status": "Matched"
            })

    return pd.DataFrame(results)


# Example input
drugs = ["aspirin", "warfarin", "metformin"]

resolved_df = resolve_drug_list(drugs, df)
resolved_df

In [ ]:
# =========================
# 5. FEATURE EXTRACTION
# =========================

def extract_drug_features(row):
    """
    Extract searchable features from a DrugBank row.
    Since the vocabulary file may not contain full interaction tables,
    this function uses available fields such as name, synonyms, description,
    and identifiers as feature signals.
    """
    if row is None:
        return {
            "name": None,
            "drugbank_id": None,
            "synonyms": set(),
            "description_terms": set(),
            "all_terms": set()
        }

    name = row[name_col]
    drugbank_id = row[drugbank_id_col] if drugbank_id_col else None

    synonym_text = row["clean_synonyms"] if "clean_synonyms" in row else ""
    description_text = row["clean_description"] if "clean_description" in row else ""

    synonyms = set(synonym_text.split())
    description_terms = set(description_text.split())

    all_terms = set(clean_text(name).split()) | synonyms | description_terms

    return {
        "name": name,
        "drugbank_id": drugbank_id,
        "synonyms": synonyms,
        "description_terms": description_terms,
        "all_terms": all_terms
    }


def get_features_for_drugs(drug_list, df):
    """
    Return a dictionary of extracted features for matched drugs.
    """
    features = {}

    for drug in drug_list:
        match = match_drug(drug, df)
        features[drug] = extract_drug_features(match)

    return features


drug_features = get_features_for_drugs(drugs, df)
drug_features

In [ ]:
# =========================
# 6. HEURISTIC INTERACTION CHECKER
# =========================

def calculate_similarity(set1, set2):
    """
    Jaccard similarity between two sets.
    """
    if not set1 or not set2:
        return 0

    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))

    if union == 0:
        return 0

    return intersection / union


def classify_interaction_risk(similarity_score):
    """
    Classify interaction risk using a transparent heuristic.
    This is not a clinical decision tool.
    """
    if similarity_score >= 0.20:
        return "High potential similarity signal"
    elif similarity_score >= 0.10:
        return "Moderate potential similarity signal"
    elif similarity_score > 0:
        return "Low potential similarity signal"
    else:
        return "No signal detected"


def check_pairwise_interactions(drug_features):
    """
    Compare every pair of drugs and detect potential interaction signals.
    """
    results = []

    for drug1, drug2 in combinations(drug_features.keys(), 2):
        f1 = drug_features[drug1]
        f2 = drug_features[drug2]

        similarity = calculate_similarity(f1["all_terms"], f2["all_terms"])
        shared_terms = sorted(list(f1["all_terms"].intersection(f2["all_terms"])))

        risk = classify_interaction_risk(similarity)

        results.append({
            "drug_1_input": drug1,
            "drug_1_matched": f1["name"],
            "drug_1_drugbank_id": f1["drugbank_id"],
            "drug_2_input": drug2,
            "drug_2_matched": f2["name"],
            "drug_2_drugbank_id": f2["drugbank_id"],
            "similarity_score": round(similarity, 4),
            "shared_terms_count": len(shared_terms),
            "shared_terms": ", ".join(shared_terms[:25]),
            "interaction_signal": risk
        })

    return pd.DataFrame(results)


interaction_df = check_pairwise_interactions(drug_features)
interaction_df

In [ ]:
# =========================
# 7. AI-STYLE SUMMARY GENERATOR
# =========================

def generate_plain_language_summary(interaction_df):
    """
    Generate a simple plain-language summary.
    This does not call an external LLM.
    It creates structured, readable summaries from the pipeline results.
    """
    summaries = []

    for _, row in interaction_df.iterrows():
        drug1 = row["drug_1_matched"] or row["drug_1_input"]
        drug2 = row["drug_2_matched"] or row["drug_2_input"]
        signal = row["interaction_signal"]
        score = row["similarity_score"]
        shared_count = row["shared_terms_count"]

        summary = (
            f"{drug1} and {drug2}: {signal}. "
            f"The pair had a similarity score of {score} based on vocabulary overlap, "
            f"with {shared_count} shared descriptive terms. "
            f"This result should be interpreted as a screening signal, not a confirmed clinical DDI."
        )

        summaries.append(summary)

    return summaries


interaction_df["plain_language_summary"] = generate_plain_language_summary(interaction_df)
interaction_df[["drug_1_matched", "drug_2_matched", "interaction_signal", "plain_language_summary"]]

In [ ]:
# =========================
# 8. SAVE CSV OUTPUTS
# =========================

resolved_path = os.path.join(OUTPUT_DIR, "resolved_drugs.csv")
interactions_path = os.path.join(OUTPUT_DIR, "drug_interactions.csv")

resolved_df.to_csv(resolved_path, index=False)
interaction_df.to_csv(interactions_path, index=False)

print("Saved:")
print(resolved_path)
print(interactions_path)

In [ ]:
# =========================
# 9. NETWORK VISUALIZATION
# =========================

def create_interaction_network(interaction_df, output_dir):
    """
    Create a network graph where nodes are drugs and edges are interaction signals.
    """
    G = nx.Graph()

    for _, row in interaction_df.iterrows():
        drug1 = row["drug_1_matched"] or row["drug_1_input"]
        drug2 = row["drug_2_matched"] or row["drug_2_input"]

        score = row["similarity_score"]
        signal = row["interaction_signal"]

        G.add_node(drug1)
        G.add_node(drug2)
        G.add_edge(drug1, drug2, weight=score, label=signal)

    plt.figure(figsize=(10, 7))
    pos = nx.spring_layout(G, seed=42)

    edge_weights = [G[u][v]["weight"] * 10 + 1 for u, v in G.edges()]

    nx.draw_networkx_nodes(G, pos, node_size=1800)
    nx.draw_networkx_edges(G, pos, width=edge_weights)
    nx.draw_networkx_labels(G, pos, font_size=10)

    plt.title("Drug Interaction Signal Network")
    plt.axis("off")

    output_path = os.path.join(output_dir, "drug_interaction_network.png")
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    return output_path


network_path = create_interaction_network(interaction_df, OUTPUT_DIR)
print("Network saved to:", network_path)

In [ ]:
# =========================
# 10. SUMMARY REPORT
# =========================

def create_summary_report(drugs, resolved_df, interaction_df, output_dir):
    """
    Create a text summary report.
    """
    report_lines = []

    report_lines.append("Drug Interaction Checker Summary Report")
    report_lines.append("=" * 45)
    report_lines.append("")
    report_lines.append("Input drugs:")
    report_lines.append(", ".join(drugs))
    report_lines.append("")

    report_lines.append("Resolved drug names:")
    for _, row in resolved_df.iterrows():
        report_lines.append(
            f"- {row['input_drug']} -> {row['matched_name']} "
            f"({row['drugbank_id']}) [{row['match_status']}]"
        )

    report_lines.append("")
    report_lines.append("Pairwise interaction signals:")
    for _, row in interaction_df.iterrows():
        report_lines.append(
            f"- {row['drug_1_matched']} + {row['drug_2_matched']}: "
            f"{row['interaction_signal']} "
            f"(similarity score: {row['similarity_score']})"
        )

    report_lines.append("")
    report_lines.append("Important note:")
    report_lines.append(
        "This pipeline is a screening and research tool. It does not replace pharmacist, "
        "physician, or clinical decision support review."
    )

    output_path = os.path.join(output_dir, "summary_report.txt")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    return output_path


summary_path = create_summary_report(drugs, resolved_df, interaction_df, OUTPUT_DIR)
print("Summary report saved to:", summary_path)

In [ ]:
# =========================
# 11. FULL PIPELINE FUNCTION
# =========================

def run_drug_interaction_checker(drug_list, drugbank_csv_path, output_dir):
    """
    Run the entire pipeline in one function.
    """
    os.makedirs(output_dir, exist_ok=True)

    df = load_drugbank_csv(drugbank_csv_path)

    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    global name_col, drugbank_id_col, synonyms_col, description_col
    name_col = find_column(df, ["name", "drug_name", "common_name"])
    drugbank_id_col = find_column(df, ["drugbank_id", "drugbankid", "id"])
    synonyms_col = find_column(df, ["synonyms", "synonym"])
    description_col = find_column(df, ["description"])

    if name_col is None:
        raise ValueError("Could not find a drug name column.")

    df["clean_name"] = df[name_col].apply(clean_text)

    if synonyms_col:
        df["clean_synonyms"] = df[synonyms_col].apply(clean_text)
    else:
        df["clean_synonyms"] = ""

    if description_col:
        df["clean_description"] = df[description_col].apply(clean_text)
    else:
        df["clean_description"] = ""

    resolved_df = resolve_drug_list(drug_list, df)
    features = get_features_for_drugs(drug_list, df)
    interaction_df = check_pairwise_interactions(features)
    interaction_df["plain_language_summary"] = generate_plain_language_summary(interaction_df)

    resolved_path = os.path.join(output_dir, "resolved_drugs.csv")
    interactions_path = os.path.join(output_dir, "drug_interactions.csv")

    resolved_df.to_csv(resolved_path, index=False)
    interaction_df.to_csv(interactions_path, index=False)

    network_path = create_interaction_network(interaction_df, output_dir)
    summary_path = create_summary_report(drug_list, resolved_df, interaction_df, output_dir)

    print("Pipeline completed.")
    print("Resolved drugs:", resolved_path)
    print("Interactions:", interactions_path)
    print("Network:", network_path)
    print("Summary:", summary_path)

    return resolved_df, interaction_df


# Run full pipeline
drugs = ["aspirin", "warfarin", "metformin"]

resolved_results, interaction_results = run_drug_interaction_checker(
    drug_list=drugs,
    drugbank_csv_path=DRUGBANK_CSV_PATH,
    output_dir=OUTPUT_DIR
)

interaction_results